### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="churn",
    dataset_year="2005",
    domain_str="technology & internet",
    # Data Source
    dataset_source="OpenML",
    original_dataset_source_download_link="https://www.openml.org/d/40701",
    download_description="""
We download the data from OpenML.

In this notebook, run:

    import openml

    # Load the dataset object from OpenML
    dataset = openml.datasets.get_dataset(
        40701,
        download_data=True,
        download_qualities=False,
        download_features_meta_data=False,
        )
    dataset.get_data()[0].to_csv("../../../local-data-warehouse/churn/churn.csv", index=False)
""",
    # References
    academic_reference_bibtex="""@misc{marcoulides2005churn,
  title={Discovering knowledge in data: An introduction to data mining},
  author={Marcoulides, George A},
  year={2005},
  publisher={Taylor \& Francis}
}
""",
    academic_reference_bibtex_key="marcoulides2005churn",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
- The original source is lost, so we use https://www.openml.org/d/40701 (or https://github.com/EpistasisLab/pmlb/tree/master/datasets/churn)
- We dropped the "phone_number" feature as it seems to be an index in the original data. 
- We renamed the target variable to "CustomerChurned" and mapped binary variables to "Yes"/"No"
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="CustomerChurned",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="CustomerChurned",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/churn.csv")

feature_names = [
    "state",
    "account_length",
    "area_code",
    "phone_number",
    "international_plan",
    "voice_mail_plan",
    "number_vmail_messages",
    "total_day_minutes",
    "total_day_calls",
    "total_day_charge",
    "total_eve_minutes",
    "total_eve_calls",
    "total_eve_charge",
    "total_night_minutes",
    "total_night_calls",
    "total_night_charge",
    "total_intl_minutes",
    "total_intl_calls",
    "total_intl_charge",
    "number_customer_service_calls",
    "class"
]

df.rename(columns={"class": "CustomerChurned"}, inplace=True)
df.drop(columns=["phone_number"], inplace=True)

cat_features = [
    "state",
    "area_code",
    "international_plan",
    "voice_mail_plan",
    "CustomerChurned"
]

df["CustomerChurned"] = df["CustomerChurned"].map({1: "Yes", 0: "No"})
df["international_plan"] = df["international_plan"].map({1: "Yes", 0: "No"})
df["voice_mail_plan"] = df["voice_mail_plan"].map({1: "Yes", 0: "No"})

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category")

In [3]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

,state,account_length,area_code,international_plan,voice_mail_plan,number_vmail_messages,total_day_minutes,total_day_calls,total_day_charge,total_eve_minutes,total_eve_calls,total_eve_charge,total_night_minutes,total_night_calls,total_night_charge,total_intl_minutes,total_intl_calls,total_intl_charge,number_customer_service_calls,CustomerChurned
0,3,72,510,No,No,0,272.4,88,46.31,107.9,125,9.17,185.5,81,8.35,12.7,2,3.43,0,No
1,23,53,415,No,No,0,164.1,106,27.90,206.0,56,17.51,194.7,124,8.76,11.4,2,3.08,1,No
2,36,155,408,No,Yes,30,61.6,103,10.47,255.1,110,21.68,225.9,96,10.17,12.4,5,3.35,1,No
3,37,161,415,No,No,0,178.1,109,30.28,146.5,86,12.45,137.6,78,6.19,8.5,2,2.30,1,No
4,11,99,415,No,No,0,62.9,81,10.69,231.0,64,19.64,168.9,121,7.60,8.5,5,2.30,1,No


## Data Checks

In [4]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 5,000
Columns: 20
Use sampling: False (sample size: 5,000)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['total_day_minutes', 'total_day_charge', 'total_eve_minutes', 'total_night_minutes', 'total_eve_charge', 'total_night_charge', 'account_length', 'total_intl_charge', 'total_intl_minutes', 'total_night_calls']
Rows remaining as candidates after top-10 filter: 0 (of 5,000)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [5]:
# Sample Rows
df_head

,state,account_length,area_code,international_plan,voice_mail_plan,number_vmail_messages,total_day_minutes,total_day_calls,total_day_charge,total_eve_minutes,total_eve_calls,total_eve_charge,total_night_minutes,total_night_calls,total_night_charge,total_intl_minutes,total_intl_calls,total_intl_charge,number_customer_service_calls,CustomerChurned
0,3,72,510,No,No,0,272.4,88,46.31,107.9,125,9.17,185.5,81,8.35,12.7,2,3.43,0,No
1,23,53,415,No,No,0,164.1,106,27.90,206.0,56,17.51,194.7,124,8.76,11.4,2,3.08,1,No
2,36,155,408,No,Yes,30,61.6,103,10.47,255.1,110,21.68,225.9,96,10.17,12.4,5,3.35,1,No
3,37,161,415,No,No,0,178.1,109,30.28,146.5,86,12.45,137.6,78,6.19,8.5,2,2.30,1,No
4,11,99,415,No,No,0,62.9,81,10.69,231.0,64,19.64,168.9,121,7.60,8.5,5,2.30,1,No


In [6]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,state,category,0.0,0.0,51.0,"49, 23, 1, 13, 45, 35, 43, 50, 37, 34"
1,area_code,category,0.0,0.0,3.0,"415, 408, 510"
2,international_plan,category,0.0,0.0,2.0,"No, Yes"
3,voice_mail_plan,category,0.0,0.0,2.0,"No, Yes"
4,CustomerChurned,category,0.0,0.0,2.0,"No, Yes"
5,total_day_minutes,float64,0.0,0.0,1961.0,"154.0, 189.3, 180.0, 159.5, 177.1, 174.5, 184.5, 183.4, 168.6, 189.8"
6,total_day_charge,float64,0.0,0.0,1961.0,"26.18, 32.18, 30.6, 27.12, 30.11, 29.67, 31.37, 31.18, 28.66, 32.27"
7,total_eve_minutes,float64,0.0,0.0,1879.0,"169.9, 199.7, 230.9, 194.0, 210.6, 187.5, 223.5, 167.6, 161.7, 188.8"
8,total_eve_charge,float64,0.0,0.0,1659.0,"15.9, 14.25, 16.12, 16.97, 18.79, 18.96, 19.41, 18.62, 16.35, 16.41"
9,total_night_minutes,float64,0.0,0.0,1853.0,"186.2, 188.2, 194.3, 214.6, 208.9, 228.1, 193.6, 169.4, 192.7, 197.4"


In [7]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
account_length,5000.0,100.258600,39.694560,1.0,243.00
number_vmail_messages,5000.0,7.755200,13.546393,0.0,52.00
total_day_minutes,5000.0,180.288900,53.894699,0.0,351.50
total_day_calls,5000.0,100.029400,19.831197,0.0,165.00
total_day_charge,5000.0,30.649668,9.162069,0.0,59.76
total_eve_minutes,5000.0,200.636560,50.551309,0.0,363.70
total_eve_calls,5000.0,100.191000,19.826496,0.0,170.00
total_eve_charge,5000.0,17.054322,4.296843,0.0,30.91
total_night_minutes,5000.0,200.391620,50.527789,0.0,395.00
total_night_calls,5000.0,99.919200,19.958686,0.0,175.00


In [8]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column             rank                    
CustomerChurned    1       No   4293  85.86
                   2      Yes    707  14.14
area_code          1      415   2495  49.90
                   2      408   1259  25.18
                   3      510   1246  24.92
international_plan 1       No   4527  90.54
                   2      Yes    473   9.46
state              1       49    158   3.16
                   2       23    125   2.50
                   3        1    124   2.48
                   4       13    119   2.38
                   5       45    118   2.36
voice_mail_plan    1       No   3677  73.54
                   2      Yes   1323  26.46

In [9]:
# Target Distribution
target_df

,count,pct
CustomerChurned,,
No,4293,85.86
Yes,707,14.14


## Task Curation

In [10]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [11]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [12]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to churn/019d9cd1-09df-75e4-a0b5-57923a0accc8
019d9cd1-09df-75e4-a0b5-57923a0accc8
81eb262a9e5bc1a1c097949d10de2ebd31df8c8b116fd52820c24192707a7428
